# Mask-DINO Swin — обучение на Kaggle GPU

Secrets: `GIT_URL`, `GITHUB_TOKEN`, `DAGSHUB_TOKEN`.

Accelerator: **GPU** (T4 ×2 или P100).

Add Data: competition `filament-segmentation-2026`.
После train — Save Version **с Output**, `last.pt`.

In [ ]:
# !rm -rf /kaggle/working/filament-segmentation-2026
# !rm -rf /kaggle/working/outputs

In [ ]:
import os, sys, subprocess
from pathlib import Path

os.chdir("/kaggle/working")

try:
    from kaggle_secrets import UserSecretsClient
    sec = UserSecretsClient()
    for name in ("GIT_URL", "GITHUB_TOKEN"):
        try:
            os.environ[name] = sec.get_secret(name)
        except Exception:
            pass
except Exception:
    pass

REPO = Path("/kaggle/working/filament-segmentation-2026")
if REPO.exists():
    import shutil
    shutil.rmtree(REPO)

url = os.environ.get("GIT_URL", "")
token = os.environ.get("GITHUB_TOKEN", "")
if token and url.startswith("https://"):
    url = url.replace("https://", f"https://{token}@", 1)

subprocess.check_call(["git", "clone", "--depth", "1", "--branch", "feature/mask-dino", url, str(REPO)])

sys.path.insert(0, str(REPO))
os.chdir(REPO)

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["DAGSHUB_TOKEN"] = UserSecretsClient().get_secret("DAGSHUB_TOKEN")
except Exception:
    pass

with open("/kaggle/working/filament-segmentation-2026/.env", "w") as f:
    f.write("DAGSHUB_REPO_OWNER=BraveCactus\n")
    f.write("DAGSHUB_REPO_NAME=solar-filament-segmentation\n")
    f.write("DAGSHUB_EXPERIMENT_NAME=mask_dino_swin\n")
    f.write(f"DAGSHUB_TOKEN={os.environ.get('DAGSHUB_TOKEN', '')}\n")

In [ ]:
!pip install uv # если не установлен пакетный менеджер uv

In [ ]:
!uv run run.py mask_dino_swin train --smoke --eval-samples-val 4

In [ ]:
!uv run run.py mask_dino_swin train --size 512 --epochs 8 --run-name "mask_dino_swin_512_8"

In [ ]:
!uv run run.py mask_dino_swin infer --submit